# EMODnet Bathymetry — European high-resolution DTM (OGC WCS)

Fetch a small **EMODnet Bathymetry** Digital Terrain Model (DTM) subset over a
North Sea AOI, map the sea-floor depth, and see how the European DTM complements
the global GEBCO / ETOPO grids. Unlike the ERDDAP `griddap` DEMs, EMODnet is read
over **OGC WCS** through `pyramids.Dataset.from_wcs` — `earthlens` supplies only the
coverage id, AOI, CRS, and protocol version, and never touches a competing array
stack.

## Setup

`earthlens` provides the unified `EarthLens` entry point; `pyramids` reads the
written GeoTIFF; downloads go to a temporary directory.

In [ ]:
import tempfile

import numpy as np
from pyramids.dataset import Dataset

from earthlens.bathymetry import Catalog
from earthlens.core import EarthLens

## The EMODnet rows in the catalog

The `bathymetry` catalog carries the latest EMODnet release as `emodnet` plus
year-stamped older releases. Each is a `wcs`-transport row pointing at the EMODnet
Bathymetry OGC WCS. Reading the catalog is offline.

In [ ]:
catalog = Catalog()
for dataset_id in sorted(d for d in catalog.datasets if d.startswith('emodnet')):
    row = catalog.get(dataset_id)
    print(f'{dataset_id:14} transport={row.transport:4} coverage={row.dataset_id}')

## Download an EMODnet subset over WCS

Build the request — the `bathymetry` source, the `emodnet` dataset, and a small
North Sea AOI (`aoi` is `[min_lon, min_lat, max_lon, max_lat]`). `download()` issues
the WCS `GetCoverage`, crops to the AOI, and returns the written GeoTIFF path(s).

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='bathymetry',
    dataset='emodnet',
    aoi=[2.0, 53.0, 4.0, 55.0],
    path=out,
).download(progress_bar=False)
paths

## Read and map the sea floor

Read the GeoTIFF with `pyramids` and map the mean depth — every value in this
all-water AOI is below sea level, so the depths are negative (metres).

In [ ]:
ds = Dataset.read_file(str(paths[0]))
arr = np.asarray(ds.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
print(
    'epsg',
    ds.epsg,
    '| shape',
    arr.shape,
    '| depth range',
    round(float(np.nanmin(arr))),
    '..',
    round(float(np.nanmax(arr))),
    'm',
)

In [ ]:
# plot() draws the Dataset itself: georeferenced axes, and
# the band's declared no-data honoured without masking.
glyph = ds.plot(
    cmap='Blues_r', title='EMODnet Bathymetry DTM (~3.75″) — southern North Sea'
)
glyph.cbar.set_label('mean depth (m, negative = below sea level)')

## European coverage — when to reach for a global DEM

EMODnet covers European seas and the extended NE-Atlantic domain, advertised as
the `native_bbox` below (`west, south, east, north`). A request whose bbox extends
beyond it warns that out-of-coverage cells come back as `0.0` fill, and a request
entirely outside raises — pointing you at the global `gebco_2020` / `etopo1_ice`
DEMs for those areas.

In [ ]:
row = catalog.get('emodnet')
print('coverage extent (W, S, E, N):', row.native_bbox)
print('for areas outside it, use dataset="gebco_2020" or "etopo1_ice"')

## Attribution

- **EMODnet Bathymetry** — EMODnet Digital Bathymetry (DTM 2024), EMODnet
  Bathymetry Consortium (doi:10.12770/cf51df64-56f9-4a99-b1aa-36b8d7b743a1).

Provided under the EMODnet use conditions (attribution required). A DEM is a static
grid with no time axis, so `download(aggregate=...)` is rejected.